### Taxi - v3 RL solution ###


In [1]:
import gymnasium as gym

import numpy as np
import random
from IPython.display import clear_output
from time import sleep

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Setting up an environment

env = gym.make("Taxi-v4",render_mode="rgb_array")

In [3]:
# Set up the Q-table

q_table = np.zeros([env.observation_space.n, env.action_space.n])
print(f"Q-Table size: {q_table.shape}")

Q-Table size: (500, 6)


In [4]:
%%time
"""Training the agent"""

frames = []

# Hyperparameters
alpha = 0.1
gamma = 0.6
epsilon = 0.1

# For plotting metrics
all_epochs = []
all_penalties = []

print("Training in progress...")
for i in range(1, 100):
    state = env.reset()[0]

    epochs, penalties, reward, = 0, 0, 0
    done = False
    
    while not done:
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Explore action space
        else:
            action = np.argmax(q_table[state]) # Exploit learned values

        next_state, reward, done, _, _ = env.step(action) 
        
        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])
        
        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        q_table[state, action] = new_value

        if reward == -10:
            penalties += 1
        
        # Put each rendered frame into dict for animation
        frames.append({
            'frame': env.render(),
            'state': state,
            'action': action,
            'reward': reward
        })
        state = next_state
        epochs += 1
        
        
    if i % 100 == 0:
        clear_output(wait=True)
        print(f"Episode: {i}")
        

print("Training finished.\n")


Training in progress...
Training finished.

CPU times: user 2min 33s, sys: 3.57 s, total: 2min 36s
Wall time: 2min 36s


In [5]:
# Run the frames to see how the agent did over the episodes

def print_frames(frames):
    for i, frame in enumerate(frames):
        clear_output(wait=True)
        print(frame['frame'])
        print(f"Timestep: {i + 1}")
        print(f"State: {frame['state']}")
        print(f"Action: {frame['action']}")
        print(f"Reward: {frame['reward']}")
        sleep(.1)
        
print_frames(frames)

[[[110 109 106]
  [110 109 106]
  [124 122 122]
  ...
  [108 111 109]
  [118 119 119]
  [118 119 119]]

 [[110 109 106]
  [110 109 106]
  [124 122 122]
  ...
  [108 111 109]
  [118 119 119]
  [118 119 119]]

 [[114 116 115]
  [114 116 115]
  [126 127 126]
  ...
  [112 113 111]
  [118 117 115]
  [118 117 115]]

 ...

 [[116 115 116]
  [116 115 116]
  [106 107 108]
  ...
  [113 115 114]
  [117 114 117]
  [117 114 117]]

 [[115 112 112]
  [115 112 112]
  [119 119 117]
  ...
  [123 119 118]
  [114 114 117]
  [114 114 117]]

 [[115 112 112]
  [115 112 112]
  [119 119 117]
  ...
  [123 119 118]
  [114 114 117]
  [114 114 117]]]
Timestep: 1421
State: 438
Action: 4
Reward: -10


KeyboardInterrupt: 

In [ ]:
"""Evaluate agent's performance after Q-learning"""

total_epochs, total_penalties = 0, 0
episodes = 100
framed = []

for _ in range(episodes):
    state = env.reset()[0]
    epochs, penalties, reward = 0, 0, 0
    
    done = False
    
    while not done:
        action = np.argmax(q_table[state])
        state, reward, done, info = env.step(action)

        if reward == -10:
            penalties += 1
        
        # Put each rendered frame into dict for animation
        framed.append({
            'frame': env.render(mode='ansi'),
            'state': state,
            'action': action,
            'reward': reward
        })
        print_frames(framed)
        epochs += 1

    total_penalties += penalties
    total_epochs += epochs

print(f"Results after {episodes} episodes:")
print(f"Average timesteps per episode: {total_epochs / episodes}")
print(f"Average penalties per episode: {total_penalties / episodes}")
